In [1]:
#!pip install pandas
#!pip install plotly
#%pip install 

import pandas as pd
import plotly.express as px

In [2]:
file_path = "C:/Users/leopi/Il mio Drive/Desktop/Magistrale/IoT_ESP_SleepSense/MisurazioniRAW/2025-12-13_12_29_influxdb_data.csv"
save_path = "2025-12-13.csv"

df_clean = pd.read_csv(file_path, skiprows=3)

# rimuovo colonne indesiderate (se presenti)
df_clean = df_clean.drop(columns=[col for col in df_clean.columns if "Unnamed" in col or col in ["result", "table", "_start", "_stop", "is_moving"]], errors='ignore')

pd.set_option("display.expand_frame_repr", False)  # niente wrapping
pd.set_option("display.max_columns", None)         # mostra tutte le colonne

# Visulaizzo il database per intero tramite pandas (utile per debug)
print(df_clean)

pd.reset_option("display.expand_frame_repr")
pd.reset_option("display.max_columns")

                    _time _measurement          host location    room   accel_x   accel_y   accel_z    gyro_x    gyro_y    gyro_z   humidity  light        mic  temperature
0    2025-12-13T02:00:05Z    corso_IoT  ESP_LEOPIZZI    Lecce  Stanza  0.317143 -0.202857  0.940000 -1.061429  2.051429  0.998571  72.514286      0   7.857143    17.500000
1    2025-12-13T02:00:40Z    corso_IoT  ESP_LEOPIZZI    Lecce  Stanza  0.318919 -0.202973  0.936757 -0.820270  2.107027  0.770811  72.213514      0   3.891892    17.600000
2    2025-12-13T02:01:15Z    corso_IoT  ESP_LEOPIZZI    Lecce  Stanza  0.318409 -0.201364  0.937045 -0.865227  2.309318  0.793182  71.877273      0   8.886364    17.636364
3    2025-12-13T02:01:50Z    corso_IoT  ESP_LEOPIZZI    Lecce  Stanza  0.316889 -0.203556  0.937778 -0.914222  2.289778  0.841111  71.615556      0   8.911111    17.728889
4    2025-12-13T02:02:25Z    corso_IoT  ESP_LEOPIZZI    Lecce  Stanza  0.317273 -0.202500  0.937273 -0.857955  2.371818  0.760227  71.315909

In [ ]:
# Calcolo is_moving
acc_treashold = 0.5
gyro_treashold = 2
num_rows = 10
# Tempo di riferimento: ~6 min (35 sec x 10 righe)

# Prendo i primi num_rows valori di accel_x, accel_y, accel_z
rif_acc = df_clean[["accel_x", "accel_y", "accel_z"]].head(num_rows)
# Faccio la media per ogni valore
rif_acc = rif_acc.mean()
# Radice quadrata del quadrato di ogni valore
rif_acc = (rif_acc**2).sum()**0.5
# Deviazione standard
rif_acc = abs(rif_acc - 1)
print("Riferimento accelerometro:", rif_acc)

# Prendo i primi num_rows valori di gyro_x, gyro_y, gyro_z
rif_gyro = df_clean[["gyro_x", "gyro_y", "gyro_z"]].head(num_rows)
rif_gyro = rif_gyro.mean()
print("Riferimento giroscopio:", rif_gyro)

# Aggiungo una colonna is_moving
df_clean["is_moving"] = 0

# Calcolo is_moving a partire dalla 6a riga

for i in range(num_rows, len(df_clean)):
    # Copio la riga corrente
    new_row =  df_clean.iloc[i, :].copy()

    # Calcolo deviazione accelerometro
    new_acc = new_row["accel_x"]**2 + new_row["accel_y"]**2 + new_row["accel_z"]**2
    new_acc = new_acc**0.5
    new_acc = abs(new_acc - 1)
    dev_acc = abs(new_acc - rif_acc)

    # Calcolo deviazione giroscopio
    dev_gyro_x = abs(new_row["gyro_x"] - rif_gyro["gyro_x"])
    dev_gyro_y = abs(new_row["gyro_y"] - rif_gyro["gyro_y"])
    dev_gyro_z = abs(new_row["gyro_z"] - rif_gyro["gyro_z"])

    if dev_acc > acc_treashold or dev_gyro_x > gyro_treashold or dev_gyro_y > gyro_treashold or dev_gyro_z > gyro_treashold:
        df_clean.at[i, "is_moving"] = 100


# Pulizia e conversione tipi
df_clean["_time"] = pd.to_datetime(df_clean["_time"], utc=True, errors="coerce", format="mixed")
cols_to_numeric = ["humidity", "light", "mic", "temperature", "is_moving", "accel_x", "accel_y", "accel_z", "gyro_x", "gyro_y", "gyro_z"]
df_clean[cols_to_numeric] = df_clean[cols_to_numeric].apply(pd.to_numeric, errors="coerce")

df_clean.to_csv(save_path, index=False)

fig = px.line(df_clean, x="_time", y=cols_to_numeric, title="Plot delle variabili nel tempo")
fig.update_layout(xaxis_title="Time", yaxis_title="Sensor Values")
fig.show()

Riferimento accelerometro: 0.010413573588514025
Riferimento giroscopio: gyro_x   -0.888091
gyro_y    2.274809
gyro_z    0.801436
dtype: float64
